# Test 2: modelos locales y longitud

Probamos Qwen2.5 de 1,5B, Gemma 2 de 2B y Qwen3 de 4B con MLX, cuantizados a 4 bits. Los pesos quedan fijos; no se cargan los adaptadores del trabajo de NLP.

Este notebook lee las ejecuciones guardadas. **Abrirlo o ejecutar sus celdas no carga modelos ni llama a una API.** El código de inferencia está en [utils/local_model_test.py](utils/local_model_test.py).

## 1. Qué cambia

- **Negociador:** usa la política $H$ para responder al caso.
- **Mejorador:** usa las instrucciones $\phi$ y los errores de desarrollo para proponer otra política.
- **Actualizador:** propone nuevas instrucciones $\phi$ para la siguiente iteración.
- **Python:** evalúa y conserva una política si mejora la validación sin incumplimientos.

Son llamadas separadas al mismo modelo. Desarrollo, validación y test tienen dos casos cada uno. El mejorador solo recibe errores de desarrollo.

$H'_t=M_{\phi_t}(H_t,O,F_{dev};L)$: propuesta. Se conserva $H'_t$ si mejora; en caso contrario, sigue $H_t$.

$\phi_{t+1}=A(\phi_t,H'_t,O,F_{dev}(H'_t);L)$: instrucciones usadas en la siguiente propuesta. Su actualización no garantiza una mejora.

In [1]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

case_dir = Path.cwd()
if not (case_dir / "results").is_dir():
    case_dir = case_dir / "research/cases/case-0003-debt-negotiation"
run_names = [
    "20261031T052617Z-local-qwen",
    "20261031T052638Z-local-gemma",
    "20261031T052812Z-local-qwen3",
]
records = []
for name in run_names:
    folder = case_dir / "results" / name
    manifest = json.loads((folder / "manifest.json").read_text())
    for filename, expected in manifest.items():
        assert Path(filename).name == filename
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == expected
    records.append({
        "name": name,
        "protocol": json.loads((folder / "protocol.json").read_text()),
        "report": json.loads((folder / "report.json").read_text()),
        "calls": [json.loads(line) for line in (folder / "calls.jsonl").read_text().splitlines()],
    })

def table(headers, rows):
    head = "".join(f'<th style="text-align:left;padding:8px;border-bottom:2px solid #667085">{html.escape(str(h))}</th>' for h in headers)
    body = "".join("<tr>" + "".join(f'<td style="text-align:left;vertical-align:top;padding:8px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>' for v in row) + "</tr>" for row in rows)
    display(HTML(f'<table style="width:100%;border-collapse:collapse"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

def panel(title, text):
    display(HTML(f'<h4>{html.escape(title)}</h4><blockquote style="border-left:3px solid #667085;padding:12px;white-space:pre-wrap;overflow-wrap:anywhere">{html.escape(text)}</blockquote>'))

print("Integridad de las tres ejecuciones verificada.")

Integridad de las tres ejecuciones verificada.


## 2. Condiciones

MacBook Pro M5 Max, 48 GB de memoria unificada. Un modelo cargado por proceso. Sin llamadas a API.

- Tres propuestas como máximo, 23 llamadas y 50.000 tokens por ejecución.
- Hasta 1.024 tokens de salida; entrada más salida reservada: 4.096 tokens.
- Tope MLX de 8 GiB y 15 minutos. Temperatura 0; semilla 42.
- JSON solicitado en el prompt, sin decodificación restringida. Una respuesta inválida falla el caso; una propuesta o actualización inválida detiene la ejecución, sin reintento.
- Qwen usa roles `system` y `user`. Gemma 2 no admite `system` en su plantilla: se incluyen instrucciones y datos en `user`.
- Qwen3 usa `enable_thinking=False`. Las revisiones y versiones de librerías están en cada `protocol.json`.

La calidad $Q$ es la proporción de decisiones que coinciden con la acción y el plan esperados y pasan las reglas estructuradas. $V$ es la proporción con incumplimientos detectados. Aquí no se puntúa el significado del mensaje ni su claridad.

In [2]:
rows = []
for record in records:
    p, r = record["protocol"], record["report"]
    selected = next(item for item in r["rounds"] if item["iteration"] == r["selected_iteration"])
    test = r.get("test")
    rows.append([
        p["model"].split("/")[-1],
        "Completa" if r["status"] == "completed" else "Detenida",
        len(r["rounds"]) - 1,
        f'H{r["selected_iteration"]}',
        f'{selected["validation"]["q"]:.0%}',
        f'{test["q"]:.0%}' if test else "No ejecutado",
        r["calls"], f'{r["elapsed_seconds"]:.1f}', f'{r["peak_mlx_gb"]:.2f}',
    ])
table(["Modelo", "Ejecución", "Propuestas evaluadas", "Conservada", "Q validación", "Q test", "Llamadas", "Segundos", "Pico MLX (GB)"], rows)

Modelo,Ejecución,Propuestas evaluadas,Conservada,Q validación,Q test,Llamadas,Segundos,Pico MLX (GB)
Qwen2.5-1.5B-Instruct-4bit,Detenida,2,H0,0%,No ejecutado,16,8.1,1.73
gemma-2-2b-it-4bit,Detenida,0,H0,0%,No ejecutado,5,5.6,2.23
Qwen3-4B-4bit,Completa,3,H0,0%,50%,23,21.8,3.19


Qwen3 completó el bucle, pero ninguna propuesta fue aceptada. El test final corresponde a **H0**, que resolvió uno de los dos casos; ese resultado no proviene de la automejora. Qwen2.5 y Gemma se detuvieron antes del test.

El pico corresponde a asignaciones registradas por MLX, no a toda la memoria del proceso ni del equipo. Los tiempos incluyen la carga del modelo y excluyen la descarga.

## 3. Longitud de H y de φ

$\ell_H=|Tokenizer(H)|$ y $\ell_\phi=|Tokenizer(\phi)|$, sin tokens especiales. Por llamada se registra también la entrada completa: contrato fijo, texto editable, datos, esquema y plantilla de chat.

La tabla muestra **candidatos**, incluidos los rechazados. Contar tokens ayuda a seguir el tamaño y el consumo; no mide claridad. Cada modelo usa su tokenizer: las longitudes entre modelos no son una escala común.

In [3]:
rows = []
for record in records:
    for item in record["report"]["rounds"]:
        rows.append([
            record["report"]["model_key"], f'H{item["iteration"]}',
            item["policy_tokens"], item["strategy_tokens"],
            f'{item["validation"]["q"]:.0%}', f'{item["validation"]["v"]:.0%}',
            "Inicial" if item["iteration"] == 0 else ("Aceptada" if item["accepted"] else "Rechazada"),
        ])
table(["Modelo", "Política", "Tokens H", "Tokens φ utilizados", "Q validación", "V validación", "Selección"], rows)

Modelo,Política,Tokens H,Tokens φ utilizados,Q validación,V validación,Selección
qwen,H0,28,31,0%,50%,Inicial
qwen,H1,28,31,0%,50%,Rechazada
qwen,H2,110,110,0%,0%,Rechazada
gemma,H0,25,25,0%,100%,Inicial
qwen3,H0,28,31,0%,100%,Inicial
qwen3,H1,177,31,0%,100%,Rechazada
qwen3,H2,110,115,0%,100%,Rechazada
qwen3,H3,110,115,0%,100%,Rechazada


En Qwen3, H1 tiene 177 tokens frente a los 28 de H0. H2 y H3 tienen 110. Ninguna resolvió los casos de validación. Con esta prueba no podemos atribuir el resultado a la longitud: también cambió el contenido.

## 4. Qué propuso Qwen3

H0 permanece como política de partida porque las tres propuestas fueron rechazadas. La primera actualización cambia φ; la segunda devuelve el mismo texto. Ambas salidas se utilizan en la propuesta siguiente.

In [4]:
record = next(r for r in records if r["report"]["model_key"] == "qwen3")
for item in record["report"]["rounds"]:
    index = item["iteration"]
    label = "Política inicial del negociador" if index == 0 else f'Propuesta desde H{item["parent_iteration"]}, rechazada'
    panel(f"H{index}: {label}", item["policy"])
for update in record["report"]["strategy_updates"]:
    index = update["after_iteration"]
    panel(f"φ{index}: instrucciones del mejorador para la propuesta {index + 1}", update["after"])

## 5. Fallos observados

Se distinguen los errores de decisión de los de formato. El evaluador se mantiene fijo.

In [5]:
rows = []
for record in records:
    for call in record["calls"]:
        if call.get("error"):
            rows.append([record["report"]["model_key"], call["call"], call["role"], call["error"]])
table(["Modelo", "Llamada", "Función", "Error de formato"], rows)
record = next(r for r in records if r["report"]["model_key"] == "qwen3")
item = record["report"]["rounds"][1]
rows = []
for row in item["validation"]["rows"]:
    rows.append([row["id"], json.dumps(row["expected"], ensure_ascii=False),
                 json.dumps(row["reply"], ensure_ascii=False), ", ".join(row["violations"])])
table(["Validación de H1", "Esperado", "Respuesta", "Incumplimiento"], rows)

Modelo,Llamada,Función,Error de formato
qwen,16,strategy_update,invalid_schema
gemma,1,negotiator,invalid_schema
gemma,2,negotiator,invalid_schema
gemma,3,negotiator,invalid_schema
gemma,4,negotiator,invalid_schema
gemma,5,proposer,invalid_json


Validación de H1,Esperado,Respuesta,Incumplimiento
validation-limited,"{""action"": ""offer"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}","{""action"": ""ask"", ""debt"": ""C"", ""payments"": 2, ""installment"": 120, ""message"": ""¿Podrías pagar la deuda C en dos cuotas de 300 cada una?""}",unexpected_plan_fields
validation-infeasible,"{""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0}","{""action"": ""offer"", ""debt"": ""C"", ""payments"": 1, ""installment"": 600, ""message"": ""Ofrecemos un pago único de 600 para la deuda C. ¿Está de acuerdo con este plan?""}",exceeds_declared_capacity


## 6. Resultados iniciales

- Qwen3 de 4B permite ejecutar los tres papeles del bucle en local. En esta ejecución no mejoró la política.
- Qwen2.5 de 1,5B y Gemma de 2B cargaron, pero no completaron el protocolo por errores de salida.
- Se conserva la longitud como registro de consumo. No entra en la selección ni sustituye a Q.
- La revisión del contrato y el contraste fijo/adaptativo continúan en la sección 7.

Son seis escenarios y una ejecución por modelo. No se ha medido generalización ni el efecto aislado de φ o de la longitud.

**Antecedente técnico.** El diagnóstico del proyecto NLP correspondía al entrenamiento de mT5 con PyTorch/MPS. En esta prueba usamos inferencia con MLX. El intento `20261031T052541Z-local-qwen` falló antes de generar texto por una incompatibilidad con la salida del tokenizer de Transformers 5; se conserva aparte y no se usa como medición de tokens ni rendimiento.

## Ejecutar el protocolo actual

En Apple Silicon con Python 3.12, instalar `requirements-local.txt` en un entorno aislado. Los snapshots fijados deben existir en la caché de Hugging Face. El script no descarga pesos ni carga adaptadores.

```sh
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3 --contract explicit --strategy adaptive
```

Para la condición fija, usar `--strategy fixed`. También acepta `qwen` y `gemma`. El protocolo actual incluye el historial de desarrollo en el feedback; las primeras ejecuciones v2 no lo incluían. Cada ejecución crea evidencia nueva en `results/`; este notebook conserva las ejecuciones indicadas al inicio.

## 7. Continuación con Qwen3 de 4B

El contrato original dejaba dudas al mapear la decisión a `action`, `debt`, `payments` e `installment`. Se redactó un contrato que define cada campo. **Es un ajuste manual del harness**, fijo durante la búsqueda.

Primero se compararon ambos contratos con la misma política manual, tomada del objetivo, solo en los dos casos de desarrollo. Después se ejecutaron tres propuestas por condición:

- **B1, fija:** φ permanece igual.
- **B2, adaptativa:** dos llamadas pueden actualizar φ entre propuestas.

Al revisar el código local se detectó que faltaba el historial del caso en el feedback al mejorador. El protocolo v4 incorpora ese contexto, como ya lo hacía el test por API. Se conservan las ejecuciones v3 previas; el contraste que se muestra abajo usa v4.

Pesos, semilla, temperatura, evaluador y criterios de selección son iguales. Se mantienen tres propuestas por condición; B2 requiere dos llamadas adicionales. No se iguala el consumo efectivo de tokens.

In [6]:
followup_names = [
    "20261031T053709Z-local-qwen3-calibration",
    "20261031T053730Z-local-qwen3-explicit-fixed",
    "20261031T053758Z-local-qwen3-explicit-adaptive",
    "20261031T053849Z-local-qwen3-explicit-fixed",
    "20261031T053935Z-local-qwen3-explicit-adaptive",
]
followups = []
for name in followup_names:
    folder = case_dir / "results" / name
    manifest = json.loads((folder / "manifest.json").read_text())
    for filename, expected in manifest.items():
        assert Path(filename).name == filename
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == expected
    followups.append({"name": name,
        "protocol": json.loads((folder / "protocol.json").read_text()),
        "report": json.loads((folder / "report.json").read_text())})
calibration = followups[0]["report"]
table(["Contrato", "Política", "Desarrollo correcto", "Con incumplimientos"], [
    [item["contract"], "Manual, sin búsqueda", f'{item["development"]["q"] * 2:.0f}/2',
     f'{item["development"]["v"] * 2:.0f}/2'] for item in calibration["calibration"]])
panel("Contrato explícito usado en el contraste", followups[-1]["protocol"]["runtime"])

Contrato,Política,Desarrollo correcto,Con incumplimientos
original,"Manual, sin búsqueda",1/2,1/2
explicit,"Manual, sin búsqueda",1/2,1/2


Ambos contratos resolvieron 1/2 casos de desarrollo. Con el explícito, el mensaje ofreció dos cuotas de 300, pero los campos devolvieron un pago de 600. Aclarar el contrato no bastó para corregir esa inconsistencia.

In [7]:
comparison = [r for r in followups if r["protocol"]["protocol"].endswith("-v4")]
rows = []
for record in comparison:
    r, p = record["report"], record["protocol"]
    selected = next(item for item in r["rounds"] if item["iteration"] == r["selected_iteration"])
    rows.append(["B1 fija" if p["strategy_mode"] == "fixed" else "B2 adaptativa",
        sum(item["accepted"] for item in r["rounds"][1:]), f'H{r["selected_iteration"]}',
        f'{selected["validation"]["q"] * 2:.0f}/2', f'{r["test"]["q"] * 2:.0f}/2',
        r["calls"], r["tokens"], f'{r["elapsed_seconds"]:.1f}', f'{r["peak_mlx_gb"]:.2f}'])
table(["Condición v4", "Aceptadas", "Conservada", "Validación", "Test reutilizado", "Llamadas", "Tokens totales", "Segundos", "Pico MLX (GB)"], rows)
rows = []
for record in comparison:
    for item in record["report"]["rounds"]:
        rows.append([record["protocol"]["strategy_mode"], f'H{item["iteration"]}',
                     item["policy_tokens"], item["strategy_tokens"],
                     f'{item["validation"]["q"] * 2:.0f}/2',
                     "Inicial" if item["iteration"] == 0 else ("Aceptada" if item["accepted"] else "Rechazada")])
table(["Condición", "Política", "Tokens H", "Tokens φ", "Validación correcta", "Selección"], rows)

Condición v4,Aceptadas,Conservada,Validación,Test reutilizado,Llamadas,Tokens totales,Segundos,Pico MLX (GB)
B1 fija,0,H0,0/2,1/2,21,14593,27.3,3.19
B2 adaptativa,0,H0,0/2,1/2,23,16128,29.9,3.19


Condición,Política,Tokens H,Tokens φ,Validación correcta,Selección
fixed,H0,28,31,0/2,Inicial
fixed,H1,177,31,0/2,Rechazada
fixed,H2,177,31,0/2,Rechazada
fixed,H3,177,31,0/2,Rechazada
adaptive,H0,28,31,0/2,Inicial
adaptive,H1,177,31,0/2,Rechazada
adaptive,H2,96,116,0/2,Rechazada
adaptive,H3,96,116,0/2,Rechazada


Se conservó H0 en ambas condiciones. El 1/2 del test corresponde a esa política inicial, no a una política mejorada. B2 consumió 1.535 tokens más, aproximadamente 10,5%, sin mejorar el resultado en esta ejecución.

B1 repitió la misma propuesta: temperatura cero, misma política conservada y mismo feedback producen la misma entrada. B2 cambió φ una vez y reutilizó ese texto; la segunda actualización no lo modificó. Este control fijo todavía no incorpora un mecanismo para evitar propuestas repetidas.

In [8]:
adaptive = next(r for r in comparison if r["protocol"]["strategy_mode"] == "adaptive")
for item in adaptive["report"]["rounds"][1:]:
    panel(f'H{item["iteration"]}: propuesta con contexto de desarrollo, rechazada', item["policy"])
for update in adaptive["report"]["strategy_updates"]:
    label = "Cambió" if update["changed"] else "Mismo texto"
    panel(f'φ después de la propuesta {update["after_iteration"]}: {label}', update["after"])


## 8. Qué sigue

- Qwen3 de 4B sigue siendo una base local viable para ejecutar el bucle. Todavía no se obtuvo una política que supere la validación.
- Antes de ampliar iteraciones, conviene conseguir decisiones estructuradas coherentes y evitar propuestas repetidas bajo las mismas condiciones para B1 y B2.
- La longitud se conserva como registro. Las políticas más largas no resolvieron los casos; tampoco se aisló el efecto de su longitud.

Los escenarios ya se habían observado y se reutilizaron para depuración. Esta prueba no permite concluir generalización, importancia causal de variables ni superioridad de una estrategia. La claridad del texto queda pendiente de un diseño específico.